# 1,021개 군집에서 반납량과 유사 기상 피처를 각각 추가한 실험

각 분기의 같은 군집·시간에 대해 MLP와 GRU를 각각 두 번 학습한다. **반납량만 추가**하면 직전 1시간·3시간 반납량이, **유사 기상만 추가**하면 유사 사례의 평균 출발량과 결측 표시가 기본 입력에 더해진다. 기본 입력에는 두 버전 모두 직전 기상 관측값이 들어간다. 마지막 표는 기존 실험에서 저장된 기본·동시 추가 모델까지 합쳐 네 가지를 비교한다.

2024년 12월 군집 구성을 고정한 회고 평가다. 1~3분기 평균은 분기별 점수의 단순평균이며, 4분기는 이전 실험에서 이미 평가했다.

In [1]:
from pathlib import Path
import os
import subprocess
import sys
from tempfile import TemporaryDirectory
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "전처리/rental_aggregate.py").is_file())
OUT = ROOT / "모델"
stations = pd.read_csv(ROOT / "따릉이 대여소 비교/서울 전체 직선거리 500m 군집 배정.csv")
stations = stations.loc[stations["기준월"].eq("2024-12")].set_index("대여소번호").sort_index()
group = stations["완전연결 군집번호"]
ids = pd.Index(sorted(group.unique()), name="군집번호")
hours = pd.date_range("2023-01-01", "2025-01-01", freq="h", inclusive="left")
assert len(stations) == 2766 and group.index.is_unique and len(ids) == 1021
assert stations["완전연결 군집 내 최대거리(m)"].max() <= 500
plt.rcParams.update({"font.family": "Malgun Gothic" if os.name == "nt" else "sans-serif",
                     "axes.unicode_minus": False, "figure.dpi": 110})
print(f"{len(hours):,}시간 × {len(ids):,}개 군집 = {len(hours)*len(ids):,}개 군집·시간")

17,544시간 × 1,021개 군집 = 17,912,424개 군집·시간


## 월별 출발 건수를 군집에 배정

0분 또는 0m 대여를 제외하고 중복을 정리한 기존 전처리 집계를 사용한다. 원본 ZIP에서 다시 만들 때는 임시 폴더를 사용한다. 이미 만들어 둔 월별 집계 폴더가 있다면 환경변수 BIKE_AGGREGATE_CACHE로 지정할 수 있다. 매월 기존 서울 전체 합계와 50개 표본의 합계를 독립적으로 대조한다.

In [2]:
verified = pd.read_csv(ROOT / "전처리/월별 집계 검증.csv").set_index("월")
sample_ids = pd.read_csv(ROOT / "전처리/선정 대여소.csv").station_id
totals = np.zeros((len(hours), len(ids)), dtype=np.int32)
# 기준선에서 시차 대여량 중 예측 정시에 아직 반납이 확인되지 않은 건수를 뺀다.
late_returns = {lag: np.zeros_like(totals) for lag in (24, 168)}
# 각 출발 시간에 대해 1~24시간 후까지 확인된 누적 대여 건수를 저장한다.
confirmed = np.empty((len(hours), len(ids), 24), dtype=np.int16)
coverage = []
with TemporaryDirectory(prefix="cluster_demand_") as tmp:
    cached = os.environ.get("BIKE_AGGREGATE_CACHE")
    cache = Path(cached) if cached else Path(tmp)
    if not cached:
        subprocess.run([sys.executable, str(ROOT / "전처리/rental_aggregate.py"), str(cache)], check=True)
    for month, ref in verified.iterrows():
        start = hours.get_loc(pd.Timestamp(month))
        with np.load(cache / f"{month}.npz") as z:
            station_ids, counts = z["ids"], z["counts"]
        nh = counts.shape[1]
        assert nh == pd.Timestamp(month).days_in_month * 24
        assert counts.sum() == ref["집계 대여 건수"]
        assert counts[np.isin(station_ids, sample_ids)].sum() == ref["표본 50곳 대여 건수"]
        groups = group.reindex(station_ids)
        matched = groups.notna().to_numpy()
        cluster_idx = ids.get_indexer(groups[matched])
        np.add.at(totals[start:start+nh].T, cluster_idx, counts[matched])
        assigned = int(totals[start:start+nh].sum())
        assert assigned == counts[matched].sum()
        coverage.append((month, int(counts.sum()), assigned, int(counts[~matched].sum())))
        events = np.memmap(cache / f"{month}.bin", dtype="<i4", mode="r").reshape(-1, 3)
        assert len(events) == ref["필터 후 원천 행"] and np.count_nonzero(events[:,0]) == counts.sum()
        for lag, correction in late_returns.items():
            mask = (events[:,0] > 0) & (events[:,2] > events[:,1] + lag)
            delayed = events[mask, :2]
            positions = ids.get_indexer(group.reindex(delayed[:,0]))
            keep = (positions >= 0) & (delayed[:,1] + lag < len(hours))
            np.add.at(correction, (delayed[keep,1] + lag, positions[keep]), 1)
        # 반납 확인이 출발 다음 정시보다 늦은 대여만 지연 시간별로 센다.
        bins = np.zeros((26, nh, len(ids)), dtype=np.int32)
        for block_start in range(0, len(events), 250_000):
            block = events[block_start:block_start+250_000]
            delay = block[:,2] - block[:,1]
            delayed = (block[:,0] > 0) & (delay > 1)
            group_pos = ids.get_indexer(group.reindex(block[delayed,0]))
            keep = group_pos >= 0
            np.add.at(bins, (np.minimum(delay[delayed][keep],25),
                             block[delayed,1][keep] - start, group_pos[keep]), 1)
        pending = np.zeros((nh, len(ids)), dtype=np.int32)
        monthly_counts = totals[start:start+nh]
        assert monthly_counts.max() <= np.iinfo(np.int16).max
        for lag in range(24,0,-1):
            pending += bins[lag+1]
            as_known = monthly_counts - pending
            assert as_known.min() >= 0
            confirmed[start:start+nh,:,lag-1] = as_known
        del bins, block, events
demand = pd.DataFrame(totals, index=hours, columns=ids)
coverage = pd.DataFrame(coverage, columns=["월", "서울 전체", "군집 배정", "명단 외"])
assert len(coverage) == 24 and (coverage["서울 전체"] == coverage["군집 배정"] + coverage["명단 외"]).all()
annual = coverage.groupby(coverage["월"].str[:4])[["서울 전체", "군집 배정", "명단 외"]].sum()
annual["배정률(%)"] = 100 * annual["군집 배정"] / annual["서울 전체"]
display(annual.round(2))

,서울 전체,군집 배정,명단 외,배정률(%)
월,,,,
2023,43994125,42810716,1183409,97.31
2024,42688696,42219615,469081,98.90


## MLP와 GRU로 각 군집의 다음 한 시간 예측

모델별로 1,021개 군집의 데이터를 함께 학습하고, 군집번호 임베딩으로 위치별 차이를 학습한다. 두 모델의 입력은 **직전 24시간의 군집별 출발량, 전주 동일 시간 출발량, 군집번호, 달력, 직전 시각의 기온·습도·강수량·풍속과 결측 표시**다. 각 과거 출발량은 해당 예측 정시 전 반납까지 확인된 값이다. MLP는 24개 값을 펼쳐 읽고 GRU는 오래된 순서부터 24번 읽는다.

이 실행은 군집 단위의 첫 비교다. 각 모델에 같은 MSE 손실, Adam 학습률 0.001, 배치 8,192, 4에포크, 시드 42를 적용한다. 에포크 수는 사전에 고정했다. 2024년 각 분기의 직전까지 매번 처음부터 학습한다. 학습 정답은 출발 후 24시간까지 확인된 값이며 학습 마감 시각까지 이 값이 준비된 행만 사용한다. 평가는 최종 유효 출발 건수로 한다. 반납량만 추가하는 실행과 유사 기상만 추가하는 실행으로 나누며, 나머지 조건은 같다.

In [3]:
import torch
from torch import nn
import math

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH, EPOCHS, SEED = 8192, 4, 42
torch.set_num_threads(4)
torch.set_float32_matmul_precision("high")
N = len(ids)
deltas = torch.arange(24,0,-1,device=DEVICE)
np.testing.assert_array_equal(confirmed[:-24,:,23], totals[:-24] - late_returns[24][24:])
weekly = np.zeros_like(totals)
weekly[168:] = totals[:-168] - late_returns[168][168:]
assert weekly.min() >= 0 and weekly.max() <= np.iinfo(np.int16).max
past = torch.as_tensor(confirmed, device=DEVICE)
week = torch.as_tensor(weekly.astype(np.int16), device=DEVICE)
actual = torch.as_tensor(totals, device=DEVICE)
weather_names = ["temperature","humidity","precipitation","windspeed"]
wx = pd.read_csv(ROOT / "분석용 데이터/Seoul_public_bicycle.csv",parse_dates=["datetime"]).set_index("datetime")
w = wx[weather_names].reindex(hours - pd.Timedelta(hours=1)).to_numpy(dtype="float32")
fixed = np.column_stack([np.eye(24)[hours.hour], np.eye(7)[hours.weekday],
                         np.eye(12)[hours.month-1], wx.holiday.loc[hours].to_numpy()]).astype("float32")
def context_at(cutoff):
    reference = w[168:cutoff-23]
    median = np.nanmedian(reference,axis=0)
    reference = np.where(np.isnan(reference),median,reference)
    mean, std = reference.mean(axis=0), reference.std(axis=0).clip(1e-6)
    values = (np.where(np.isnan(w),median,w)-mean)/std
    combined = np.column_stack([fixed,values,np.isnan(w)])
    assert np.isfinite(combined).all()
    return torch.as_tensor(combined,device=DEVICE), {"median":median.tolist(),"mean":mean.tolist(),"std":std.tolist()}

### 최근 반납량과 유사 기상 출발량

예측 정시가 09시이면 같은 군집에 08~09시 직전, 06~09시 직전 도착한 유효 대여를 각각 합산한다. 출발지는 서울 전체이며 군집 내 이동도 목적지 군집에서 한 번 센다. 이용시간·거리·중복 처리 기준은 기존 실험과 같다. 시간대 첫 경계의 불완전한 창은 앞선 168시간을 제외하는 학습·평가 규칙에 따라 사용되지 않는다.

같은 시각·요일 유형(평일 또는 주말·공휴일)·강수 유무를 가진 지난 365일 중 기온 ±5°C, 습도 ±15%p, 강수량 ±2mm, 풍속 ±2m/s 조건을 모두 만족하는 시각을 검색한다. 네 변수의 차이를 각각 허용 차이로 나누고 제곱합이 작은 순으로 최대 5개를 택한다. 모두 직전 시각의 기상 관측값으로 비교한다. 서울 기상이 공통이어서 매칭 날짜는 같지만, 평균 출발량은 각 군집 고유의 값이다. 각 과거 출발에서 24시간 뒤까지 확인된 값만 포함하며, 매칭 실패는 별도 표시한다.

In [4]:
return_cache = ROOT / "전처리/1021개 군집 시간별 반납.npz"
if not return_cache.is_file():
    subprocess.run([sys.executable, str(ROOT / "전처리/cluster_returns.py")], check=True)
with np.load(return_cache) as cache:
    np.testing.assert_array_equal(cache["ids"],ids)
    np.testing.assert_array_equal(cache["station_ids"],group.index)
    np.testing.assert_array_equal(cache["station_groups"],group)
    returned = cache["counts"]
assert returned.shape == totals.shape and returned.min() >= 0 and returned.max() <= 32767
arrivals = torch.as_tensor(returned.astype(np.int16),device=DEVICE)

limits = np.array([5.,15.,2.,2.])  # temperature, humidity, precipitation, windspeed
assert weather_names == ["temperature","humidity","precipitation","windspeed"]
complete = np.isfinite(w).all(axis=1)
wet = w[:,2] > 0
off_day = (hours.weekday >= 5) | wx.holiday.loc[hours].eq(1).to_numpy()
weather_avg = np.full(totals.shape, np.nan, dtype=np.float32)
match_count = np.zeros(len(hours),dtype="int8")
offsets = np.arange(1,366)*24
for t in range(len(hours)):
    if not complete[t]:
        continue
    prev = t-offsets[offsets<=t]
    prev = prev[complete[prev] & (off_day[prev]==off_day[t]) & (wet[prev]==wet[t])]
    differences = np.abs(w[prev]-w[t])/limits
    close = (differences <= 1).all(axis=1)
    distances = (differences[close]**2).sum(axis=1)
    selected = prev[close][np.argsort(distances,kind="stable")[:5]]
    if len(selected):
        assert (selected+24 <= t).all()
        match_count[t] = len(selected)
        weather_avg[t] = confirmed[selected,:,23].mean(axis=0)
assert np.array_equal(np.isnan(weather_avg).all(axis=1),match_count==0)
matched_avg = torch.as_tensor(weather_avg,device=DEVICE)
missing_match = torch.as_tensor(match_count==0,device=DEVICE)
print(f"군집 도착 기록 {returned.sum():,}건")
display(pd.DataFrame({"매칭 비율":(match_count>0).astype(float)},index=hours).loc["2024"]
        .resample("QS").mean().round(4))


군집 도착 기록 84,684,782건


,매칭 비율
2024-01-01,0.8489
2024-04-01,0.9734
2024-07-01,0.9665
2024-10-01,0.9176


In [5]:
VARIANTS = {
    "returns_only": ("반납량만", ["returns_prev_1h", "returns_prev_3h"]),
    "weather_match_only": ("유사 기상만", ["weather_match_mean", "weather_match_missing"]),
}

def batch_inputs(flat, variant):
    t = flat // N
    g = flat % N
    seq = past[t[:,None]-deltas,g[:,None],deltas-1].float().log1p()
    weekly = week[t,g].float().log1p()
    if variant == "returns_only":
        last = arrivals[t-1,g].float()
        last_three = last + arrivals[t-2,g] + arrivals[t-3,g]
        historical = torch.stack([weekly,last.log1p(),last_three.log1p()],dim=1)
    elif variant == "weather_match_only":
        average = matched_avg[t,g].nan_to_num().log1p()
        historical = torch.stack([weekly,average,missing_match[t].float()],dim=1)
    else:
        raise ValueError(variant)
    return t, g, seq, historical

class ClusterNet(nn.Module):
    def __init__(self, kind, context_size, extra_size):
        super().__init__()
        self.embedding = nn.Embedding(N, 8)
        self.gru = nn.GRU(1,16,batch_first=True) if kind == "GRU" else None
        history_size = 16 if self.gru is not None else 24
        self.head = nn.Sequential(nn.Linear(history_size + context_size + 8 + 1 + extra_size, 64), nn.ReLU(),
                                  nn.Linear(64,32), nn.ReLU(), nn.Linear(32,1), nn.Softplus())

    def forward(self, seq, historical_features, context, g):
        history = seq
        if self.gru is not None:
            _, hidden = self.gru(seq.unsqueeze(-1))
            history = hidden[-1]
        features = torch.cat([history,historical_features,context,self.embedding(g)],dim=1)
        return self.head(features).squeeze(1)

print(f"학습 장치 {DEVICE}, 군집 {N:,}개")

학습 장치 cuda, 군집 1,021개


In [6]:
def train_model(kind, cutoff, context, variant):
    torch.manual_seed(SEED)
    model = ClusterNet(kind, context.shape[1], len(VARIANTS[variant][1])).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(),lr=0.001)
    loss_fn = nn.MSELoss()
    # t + 24 <= cutoff이고 전주 동일 시간 관측이 있는 모든 학습 행을 사용한다.
    n_rows = (cutoff - 23 - 168) * N
    for epoch in range(EPOCHS):
        model.train()
        order = torch.randperm(n_rows,device=DEVICE)
        for ix in order.split(BATCH):
            t,g,seq,historical_features = batch_inputs(ix + 168*N, variant)
            pred = model(seq,historical_features,context[t],g)
            loss = loss_fn(pred,past[t,g,23].float())
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(),5.)
            optimizer.step()
    return model

@torch.inference_mode()
def evaluate(model, start, stop, context, variant):
    model.eval()
    sums = torch.zeros(2,device=DEVICE,dtype=torch.float64)
    for begin in range(start*N,stop*N,BATCH):
        flat = torch.arange(begin,min(stop*N,begin+BATCH),device=DEVICE)
        t,g,seq,historical_features = batch_inputs(flat, variant)
        err = (model(seq,historical_features,context[t],g) - actual[t,g]).double()
        sums += torch.stack([err.abs().sum(),err.square().sum()])
    n_rows = (stop-start)*N
    mae, mse = (sums/n_rows).cpu().tolist()
    assert math.isfinite(mae) and math.isfinite(mse)
    return n_rows, mae, math.sqrt(mse)

In [7]:
neural = []
for quarter, begin in enumerate(pd.date_range("2024-01-01","2024-10-01",freq="QS"),1):
    start = int(hours.searchsorted(begin))
    stop = int(hours.searchsorted(begin + pd.DateOffset(months=3)))
    context, scaling = context_at(start)
    for variant, (label, feature_names) in VARIANTS.items():
        for kind in ("MLP","GRU"):
            model = train_model(kind,start,context,variant)
            size,mae,rmse = evaluate(model,start,stop,context,variant)
            assert size == (stop-start)*N
            neural.append([quarter,kind,label,size,mae,rmse])
            print(f"{quarter}분기 {kind} {label} · MAE {mae:.4f}, RMSE {rmse:.4f}",flush=True)
            if quarter == 4:
                checkpoint = {"kind":kind,"weights":model.cpu().state_dict(),"cluster_ids":list(map(int,ids)),
                              "context_size":context.shape[1],"weather":weather_names,"variant":variant,
                              "extra_features":feature_names,"scaling":scaling,"seed":SEED,
                              "epochs":EPOCHS,"cutoff":str(begin)}
                torch.save(checkpoint, OUT / f"cluster1021_{kind.lower()}_{variant}.pt")
            del model
individual = pd.DataFrame(neural,columns=["분기","모델","추가 입력","군집·시간 수","MAE","RMSE"])


1분기 MLP 반납량만 · MAE 1.4179, RMSE 2.4636


1분기 GRU 반납량만 · MAE 1.4045, RMSE 2.4245


1분기 MLP 유사 기상만 · MAE 1.4897, RMSE 2.5323


1분기 GRU 유사 기상만 · MAE 1.4593, RMSE 2.4871


2분기 MLP 반납량만 · MAE 2.1535, RMSE 3.6865


2분기 GRU 반납량만 · MAE 2.1469, RMSE 3.6643


... (6 outputs left out)

4분기 MLP 반납량만 · MAE 1.7208, RMSE 2.9679


4분기 GRU 반납량만 · MAE 1.6653, RMSE 2.8618


4분기 MLP 유사 기상만 · MAE 1.6775, RMSE 2.8921


4분기 GRU 유사 기상만 · MAE 1.6899, RMSE 2.8961


In [8]:
previous = pd.read_csv(OUT/"1021개 군집 반납·유사기상 비교.csv")
tables = [individual]
for suffix, label in [("기존","없음"),("추가","둘 다")]:
    columns = {f"{metric} {suffix}":metric for metric in ["군집·시간 수","MAE","RMSE"]}
    run = previous[["분기","모델",*columns]].rename(columns=columns).assign(**{"추가 입력":label})
    tables.append(run)
results = pd.concat(tables,ignore_index=True).sort_values(["분기","모델","추가 입력"])
assert len(results) == 32 and not results.duplicated(["분기","모델","추가 입력"]).any()
assert results.groupby("분기")["군집·시간 수"].nunique().eq(1).all()
assert np.isfinite(results[["MAE","RMSE"]]).all().all()
base = results.loc[results["추가 입력"]=="없음",["분기","모델","MAE","RMSE"]]
results = results.merge(base,on=["분기","모델"],suffixes=(""," 기본"),validate="many_to_one")
for metric in ["MAE","RMSE"]:
    results[metric+" 차이"] = results[metric] - results[metric+" 기본"]
results.to_csv(OUT/"1021개 군집 피처 개별 비교.csv",index=False,encoding="utf-8-sig")
order = ["없음","반납량만","유사 기상만","둘 다"]
for metric in ["MAE","RMSE"]:
    print(metric)
    display(results.pivot(index=["모델","분기"],columns="추가 입력",values=metric)[order].round(4))
print("1~3분기 MAE 단순평균")
display(results.loc[results["분기"]<=3].pivot_table(index="모델",columns="추가 입력",values="MAE")[order].round(4))


MAE


추가 입력       없음    반납량만  유사 기상만     둘 다
모델  분기                                
GRU 1   1.4429  1.4045  1.4593  1.4311
    2   2.2122  2.1469  2.1424  2.0774
    3   2.0680  1.9826  1.9898  1.9141
    4   1.7265  1.6653  1.6899  1.6494
MLP 1   1.4603  1.4179  1.4897  1.4828
    2   2.2868  2.1535  2.1967  2.1580
    3   2.0086  1.9808  1.9498  1.9162
    4   1.7806  1.7208  1.6775  1.6472

RMSE


추가 입력       없음    반납량만  유사 기상만     둘 다
모델  분기                                
GRU 1   2.5232  2.4245  2.4871  2.4411
    2   3.8293  3.6643  3.5976  3.4806
    3   3.4969  3.4094  3.3629  3.2702
    4   2.9410  2.8618  2.8961  2.8480
MLP 1   2.5291  2.4636  2.5323  2.5003
    2   3.8826  3.6865  3.7026  3.6182
    3   3.5310  3.3857  3.3553  3.3263
    4   3.0744  2.9679  2.8921  2.8315

1~3분기 MAE 단순평균


추가 입력,없음,반납량만,유사 기상만,둘 다
모델,,,,
GRU,1.9077,1.8447,1.8638,1.8075
MLP,1.9186,1.8507,1.8787,1.8523
